# Table 1: Normative Simulacra Extraction Statistics

Regenerates the extraction stats table from `abstracted_norms.parquet` and `ci_flows.parquet`.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/share/pierson/matt/n2s4cir/data/fiction10")

norms = pd.read_parquet(DATA_DIR / "abstracted_norms.parquet")
flows = pd.read_parquet(DATA_DIR / "ci_flows.parquet")

print(f"Norms: {len(norms):,} rows, Flows: {len(flows):,} rows")

Norms: 11,869 rows, Flows: 1,241 rows


In [2]:
# Compute per-book statistics
books = sorted(norms["book_title"].unique())

rows = []
for book in books:
    n = norms[norms["book_title"] == book]
    f = flows[flows["book_title"] == book]

    # Chunks: union of unique chunk_ids across both files
    chunk_ids = set(n["chunk_id"].unique()) | set(f["chunk_id"].unique())

    rows.append({
        "Source Text": book,
        "Chunks": len(chunk_ids),
        "Flows": len(f),
        r"$|\hat{\mathcal{N}}_b|$": n["raz_norm_articulation"].nunique(),
    })

df = pd.DataFrame(rows)

In [3]:
# Add totals row
totals = df.select_dtypes("number").sum()
totals["Source Text"] = "**Total**"
df_display = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)
df_display

,Source Text,Chunks,Flows,$|\hat{\mathcal{N}}_b|$
0,Alice's Adventures in Wonderland,24,6,120
1,Anna Karenina,255,58,1304
2,Bleak House,339,210,1751
3,Les Misérables,514,255,2864
4,Middlemarch,312,128,1515
5,Nineteen Eighty-Four,108,140,588
6,Pride and Prejudice,123,49,664
7,The Age of Innocence,86,20,384
8,The Count of Monte Cristo,406,358,2041
9,The Picture of Dorian Gray,49,17,267


In [4]:
# Emit LaTeX
def fmt(x):
    """Format integers with commas."""
    return f"{int(x):,}"

COLS = ["Chunks", "Flows", r"$|\hat{\mathcal{N}}_b|$"]

header = r"""\begin{table}[ht]
\centering
\begin{tabular}{lrrr}
\toprule
Source Text & Chunks & Flows & $|\hat{\mathcal{N}}_b|$ \\
\midrule"""

footer = r"""\bottomrule
\end{tabular}
\caption{Normative simulacra extraction statistics. Chunks: 2,000-word segments with 150-word overlap. Flows: CI information flow tuples. $|\hat{\mathcal{N}}_b|$: deduplicated normative universe size after embedding-based clustering (HDBSCAN), LLM-mediated merging, and a post-hoc semantic similarity audit that reverts low-coherence merges (majority vote across three embedding models at a cosine similarity threshold of 0.5).}
\label{tab:extraction-stats}
\end{table}"""

lines = [header]
for i, row in df.iterrows():
    name = row["Source Text"]
    name_tex = name.replace("é", r"\'{e}")
    vals = " & ".join([name_tex] + [fmt(row[c]) for c in COLS])
    lines.append(f"{vals} \\\\")

# Totals
lines.append(r"\midrule")
tot = df[COLS].sum()
vals = " & ".join([r"\textbf{Total}"] + [fmt(tot[c]) for c in COLS])
lines.append(f"{vals} \\\\")
lines.append(footer)

latex = "\n".join(lines)
print(latex)

\begin{table}[ht]
\centering
\begin{tabular}{lrrr}
\toprule
Source Text & Chunks & Flows & $|\hat{\mathcal{N}}_b|$ \\
\midrule
Alice's Adventures in Wonderland & 24 & 6 & 120 \\
Anna Karenina & 255 & 58 & 1,304 \\
Bleak House & 339 & 210 & 1,751 \\
Les Mis\'{e}rables & 514 & 255 & 2,864 \\
Middlemarch & 312 & 128 & 1,515 \\
Nineteen Eighty-Four & 108 & 140 & 588 \\
Pride and Prejudice & 123 & 49 & 664 \\
The Age of Innocence & 86 & 20 & 384 \\
The Count of Monte Cristo & 406 & 358 & 2,041 \\
The Picture of Dorian Gray & 49 & 17 & 267 \\
\midrule
\textbf{Total} & 2,216 & 1,241 & 11,498 \\
\bottomrule
\end{tabular}
\caption{Normative simulacra extraction statistics. Chunks: 2,000-word segments with 150-word overlap. Flows: CI information flow tuples. $|\hat{\mathcal{N}}_b|$: deduplicated normative universe size after embedding-based clustering (HDBSCAN), LLM-mediated merging, and a post-hoc semantic similarity audit that reverts low-coherence merges (majority vote across three embedding 